# KuchoLM training — 7M copy-focused

CC100-ja から NIDA_FICTION 学習データを生成し、SentencePiece → 約7M Transformer 学習 → 評価 → 保存 → 推論までこの notebook だけで実行します。

今回の重点は **「口調を変える部分以外を壊さずコピーする」** ことです。

- 12k BPE + `byte_fallback`
- SentencePiece の正規化を抑えて入力表記を保持
- 長すぎる例は末尾を切らず、学習対象から除外
- rare Unicode / ID / 数字を両側へ同じまま挿入する copy augmentation
- 約 6.97M parameters
- exact match / 文字類似度 / REF marker copy accuracy を評価


In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece torch


## 1. 設定


In [ ]:
from pathlib import Path
import difflib
import json
import math
import random
import re
import string

import MeCab
import sentencepiece as spm
import torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import Dataset, DataLoader

DATA_PATH = Path('/content/kucholm_nida.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS = 100_000
DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'

VOCAB_SIZE = 12_000
MAX_LEN = 192
COPY_AUGMENT_RATIO = 0.20

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tagger = MeCab.Tagger()
print('device:', device)


## 2. NIDA_FICTION データ生成

元 notebook の変換ルールを保ちます。`/content/kucholm_nida.jsonl` が既にある場合は再生成しません。


In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
SENTENCE_SPLIT_RE = re.compile(r'(.+?[。！？!?]+|.+$)', re.S)

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            f = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': f[0] if len(f) > 0 else '',
                'ctype': f[4] if len(f) > 4 else '*',
                'lemma': f[7] if len(f) > 7 else '*',
                'orth_base': f[10] if len(f) > 10 else '*',
            })
        node = node.next
    return tokens

def dictionary_form(token):
    for key in ('orth_base', 'lemma'):
        value = token.get(key, '*')
        if value not in {'', '*'} and re.search(r'[ぁ-ん一-龯]', value):
            return value
    return token['surface']

def is_ichidan(token, base):
    ctype = token.get('ctype', '')
    return (
        '下一段' in ctype
        or '上一段' in ctype
        or '一段' in ctype
        or (base.endswith('る') and token.get('surface', '') == base[:-1])
    )

def ta_form(base, token):
    if base == '行く':
        return '行った'
    if base == '来る':
        return '来た'
    if base == 'する':
        return 'した'
    if is_ichidan(token, base):
        return base[:-1] + 'た'
    if base.endswith(('う', 'つ', 'る')):
        return base[:-1] + 'った'
    if base.endswith(('む', 'ぶ', 'ぬ')):
        return base[:-1] + 'んだ'
    if base.endswith('く'):
        return base[:-1] + 'いた'
    if base.endswith('ぐ'):
        return base[:-1] + 'いだ'
    if base.endswith('す'):
        return base[:-1] + 'した'
    return base + 'た'

def nai_form(base, token):
    if base == 'する':
        return 'しない'
    if base == '来る':
        return '来ない'
    if is_ichidan(token, base):
        return base[:-1] + 'ない'
    if base.endswith('う'):
        return base[:-1] + 'わない'
    table = {'く':'か', 'ぐ':'が', 'す':'さ', 'つ':'た', 'ぬ':'な', 'ぶ':'ば', 'む':'ま', 'る':'ら'}
    return base[:-1] + table[base[-1]] + 'ない' if base[-1:] in table else base + 'ない'

def soften_surface(text):
    replacements = [
        (r'ということです$', 'ってこと'),
        (r'ということでした$', 'ってことだった'),
        (r'のであります$', 'んだ'),
        (r'であります$', 'なんだ'),
        (r'なのです$', 'なんだ'),
        (r'のです$', 'んだ'),
        (r'でしょう$', 'だろう'),
        (r'ではありません$', 'じゃない'),
        (r'ではないです$', 'じゃない'),
        (r'ではない$', 'じゃない'),
    ]
    for pattern, replacement in replacements:
        text = re.sub(pattern, replacement, text)
    return text

def auxiliary_tail(body):
    replacements = [
        (r'てきちゃいました$', 'てきちゃった'),
        (r'て来ちゃいました$', 'て来ちゃった'),
        (r'てきました$', 'てきた'),
        (r'て来ました$', 'て来た'),
        (r'てこられました$', 'てこられた'),
        (r'て来られました$', 'て来られた'),
        (r'ていきました$', 'ていった'),
        (r'て行きました$', 'て行った'),
        (r'てしまいました$', 'てしまった'),
        (r'でしまいました$', 'でしまった'),
        (r'できました$', 'できた'),
    ]
    for pattern, replacement in replacements:
        if re.search(pattern, body):
            return re.sub(pattern, replacement, body)
    return None

def convert_polite_tail(body):
    replacements = [
        (r'かもしれません$', 'かもしれない'),
        (r'わかりません$', 'わからない'),
        (r'知りません$', '知らない'),
        (r'いけません$', 'いけない'),
        (r'ありません$', 'ない'),
        (r'ございました$', 'あった'),
        (r'ございます$', 'ある'),
    ]
    for pattern, replacement in replacements:
        if re.search(pattern, body):
            return re.sub(pattern, replacement, body)

    converted = auxiliary_tail(body)
    if converted is not None:
        return converted

    tokens = parse_tokens(body)
    if not tokens:
        return body

    surfaces = [token['surface'] for token in tokens]
    particle = ''
    if surfaces and surfaces[-1] in {'ね', 'よ', 'な'}:
        particle = surfaces.pop()
        tokens = tokens[:-1]

    suffixes = [
        (['ませ', 'ん', 'でし', 'た'], 'negative_past'),
        (['ませ', 'ん'], 'negative'),
        (['まし', 'た'], 'past'),
        (['ます'], 'present'),
    ]
    for suffix, mode in suffixes:
        if len(surfaces) < len(suffix) or surfaces[-len(suffix):] != suffix:
            continue

        suffix_start = len(tokens) - len(suffix)
        verb_index = next(
            (i for i in range(suffix_start - 1, -1, -1) if tokens[i]['pos'] == '動詞'),
            None,
        )
        if verb_index is None:
            continue

        verb = tokens[verb_index]
        base = dictionary_form(verb)
        prefix = ''.join(token['surface'] for token in tokens[:verb_index])
        chain = ''.join(token['surface'] for token in tokens[max(0, verb_index - 2):verb_index + 1])

        if any(x in chain for x in ('られ', 'され', 'こられ', 'おられ')):
            stem = ''.join(token['surface'] for token in tokens[:suffix_start])
            if mode == 'past':
                return stem + 'た' + particle
            if mode == 'present':
                return stem + particle

        if mode == 'present':
            replacement = base
        elif mode == 'past':
            replacement = ta_form(base, verb)
        else:
            negative = nai_form(base, verb)
            replacement = negative if mode == 'negative' else negative[:-2] + 'なかった'

        return prefix + replacement + particle

    if surfaces[-2:] == ['でし', 'た']:
        return ''.join(surfaces[:-2]) + 'だった' + particle
    if surfaces[-1:] == ['です']:
        return ''.join(surfaces[:-1]) + particle
    return body

def convert_sentence(sentence):
    match = re.match(r'^(\s*)(.*?)(\s*)$', sentence, re.S)
    leading, core, trailing = match.groups()
    if not core or URL_RE.search(core):
        return sentence

    punctuation_match = re.search(r'([。！？!?]+)$', core)
    punctuation = punctuation_match.group(1) if punctuation_match else ''
    body = core[:-len(punctuation)] if punctuation else core
    is_question = bool(re.search(r'[？?]$', punctuation))

    body = soften_surface(body)
    if body.endswith('か') and is_question:
        body = body[:-1]

    converted = convert_polite_tail(body)
    if converted.endswith(('ね', 'よ', 'な')):
        particle = converted[-1]
        converted = converted[:-1] + 'ニダ' + particle
    else:
        converted += (
            'ニカ'
            if is_question
            else ('ニダね' if re.search(r'(ない|難しい|心配|残念|大丈夫)$', converted) else 'ニダよ')
        )

    return leading + converted + punctuation + trailing

def to_nida(text):
    if not text or URL_RE.search(text):
        return None

    parts = []
    cursor = 0
    for match in SENTENCE_SPLIT_RE.finditer(text):
        if match.start() > cursor:
            parts.append(text[cursor:match.start()])
        parts.append(convert_sentence(match.group(0)))
        cursor = match.end()

    if cursor < len(text):
        parts.append(text[cursor:])
    return ''.join(parts)

if not DATA_PATH.exists():
    print('Generating NIDA_FICTION JSONL...')
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    written = 0
    with DATA_PATH.open('w', encoding='utf-8') as out:
        for row in dataset:
            source = str(row[TEXT_COLUMN])
            if not source or len(source.strip()) < 2 or len(source) > 256:
                continue

            target = to_nida(source)
            if not target or target == source:
                continue

            out.write(json.dumps(
                {'style': 'NIDA_FICTION', 'source': source, 'target': target},
                ensure_ascii=False,
            ) + '\n')
            written += 1
            if written >= MAX_ROWS:
                break

    print('written:', written)
    print('saved:', DATA_PATH)
    print('size MB:', DATA_PATH.stat().st_size / 1024 / 1024)
else:
    print('using existing:', DATA_PATH)


## 3. データ読み込み + copy augmentation

元文と正解文の両方へ同じ `[REF:...]` を挿入します。  
モデルには「ここは変換対象ではなく、そのまま保持すべき情報」として学習させます。


In [ ]:
raw_rows = []
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        raw_rows.append((item['source'], item['target']))

random.shuffle(raw_rows)
cut = max(1, int(len(raw_rows) * 0.98))
raw_train_rows = raw_rows[:cut]
raw_val_rows = raw_rows[cut:]

RARE_CHARS = '髙﨑𠮷神邉邊齋齊塚﨑'
ASCII_POOL = string.ascii_uppercase + string.digits

def make_ref():
    left = ''.join(random.choices(ASCII_POOL, k=7))
    right = ''.join(random.choices(ASCII_POOL, k=5))
    rare = ''.join(random.choices(RARE_CHARS, k=2))
    return f'[REF:{left}-{right}/{rare}]'

train_rows = []
for source, target in raw_train_rows:
    train_rows.append((f'<NIDA_FICTION> {source}', target))
    if random.random() < COPY_AUGMENT_RATIO:
        ref = make_ref()
        train_rows.append((f'<NIDA_FICTION> {ref} {source}', f'{ref} {target}'))

val_rows = [(f'<NIDA_FICTION> {source}', target) for source, target in raw_val_rows]

copy_eval_rows = []
for source, target in raw_val_rows[:200]:
    ref = make_ref()
    copy_eval_rows.append((f'<NIDA_FICTION> {ref} {source}', f'{ref} {target}', ref))

random.shuffle(train_rows)
print('train:', len(train_rows), 'val:', len(val_rows), 'copy-eval:', len(copy_eval_rows))


## 4. SentencePiece — copy重視

`byte_fallback=True` により未知文字を `<unk>` に潰さず byte 列として表現できます。  
`normalization_rule_name='identity'` で入力の表記を勝手に正規化しにくくします。


In [ ]:
spm_input = WORK_DIR / 'spm_train.txt'
with spm_input.open('w', encoding='utf-8') as f:
    for source, target in train_rows:
        f.write(source.replace('\n', ' ') + '\n')
        f.write(target.replace('\n', ' ') + '\n')

spm.SentencePieceTrainer.train(
    input=str(spm_input),
    model_prefix=str(WORK_DIR / 'kucholm_spm'),
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    character_coverage=1.0,
    byte_fallback=True,
    normalization_rule_name='identity',
    split_digits=True,
    hard_vocab_limit=False,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    user_defined_symbols=['<NIDA_FICTION>'],
)

sp = spm.SentencePieceProcessor(model_file=str(WORK_DIR / 'kucholm_spm.model'))
PAD, UNK, BOS, EOS = 0, 1, 2, 3
VOCAB = sp.vocab_size()

print('vocab:', VOCAB)
for sample in ['髙﨑𠮷野家ABC-123', 'KuchoLM-v0.7', '今日は学校です。']:
    ids = sp.encode(sample, out_type=int)
    print(sample, '->', sp.decode(ids), '| unk:', ids.count(UNK))


## 5. Dataset / DataLoader

**重要:** 旧版のように `[:MAX_LEN-2]` で末尾を切りません。  
口調変換は文末に変更が集中するので、末尾切りは学習そのものを壊します。長すぎる例は丸ごと除外します。


In [ ]:
def encode_full(text):
    return [BOS] + sp.encode(text, out_type=int) + [EOS]

def fits(text):
    return len(encode_full(text)) <= MAX_LEN

before_train = len(train_rows)
train_rows = [(source, target) for source, target in train_rows if fits(source) and fits(target)]
val_rows = [(source, target) for source, target in val_rows if fits(source) and fits(target)]
copy_eval_rows = [
    (source, target, ref)
    for source, target, ref in copy_eval_rows
    if fits(source) and fits(target)
]
print('filtered train:', before_train, '->', len(train_rows))

class PairDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        source, target = self.data[index]
        return torch.tensor(encode_full(source)), torch.tensor(encode_full(target))

def collate(batch):
    sources, targets = zip(*batch)
    return (
        nn.utils.rnn.pad_sequence(sources, batch_first=True, padding_value=PAD),
        nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=PAD),
    )

BATCH = 64 if device.type == 'cuda' else 8
train_loader = DataLoader(
    PairDataset(train_rows),
    batch_size=BATCH,
    shuffle=True,
    collate_fn=collate,
    pin_memory=device.type == 'cuda',
)
val_loader = DataLoader(
    PairDataset(val_rows),
    batch_size=BATCH,
    shuffle=False,
    collate_fn=collate,
    pin_memory=device.type == 'cuda',
)


## 6. KuchoLM NIDA-7M


In [ ]:
D_MODEL = 224
NHEAD = 8
ENC_LAYERS = 3
DEC_LAYERS = 3
FF = 896
DROPOUT = 0.1
EPOCHS = 6
LR = 3e-4

class KuchoTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.pos = nn.Embedding(MAX_LEN, D_MODEL)
        self.tf = nn.Transformer(
            d_model=D_MODEL,
            nhead=NHEAD,
            num_encoder_layers=ENC_LAYERS,
            num_decoder_layers=DEC_LAYERS,
            dim_feedforward=FF,
            dropout=DROPOUT,
            batch_first=True,
            norm_first=True,
        )
        self.lm_head = nn.Linear(D_MODEL, VOCAB, bias=False)
        self.lm_head.weight = self.embed.weight

    def add_pos(self, token_ids):
        positions = torch.arange(token_ids.size(1), device=token_ids.device).unsqueeze(0)
        return self.embed(token_ids) * math.sqrt(D_MODEL) + self.pos(positions)

    def forward(self, source, target_input):
        source_padding = source.eq(PAD)
        target_padding = target_input.eq(PAD)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            target_input.size(1),
            device=target_input.device,
        )
        hidden = self.tf(
            self.add_pos(source),
            self.add_pos(target_input),
            tgt_mask=causal_mask,
            src_key_padding_mask=source_padding,
            tgt_key_padding_mask=target_padding,
            memory_key_padding_mask=source_padding,
        )
        return self.lm_head(hidden)

model = KuchoTransformer().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'{parameter_count / 1e6:.3f}M parameters')
assert parameter_count < 7_100_000, '7M targetから大きく外れています'


## 7. 学習


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.98),
    weight_decay=0.01,
)
criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.03)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

best_val = float('inf')
best_path = WORK_DIR / 'KuchoLM-NIDA-7M.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for source, target in train_loader:
        source = source.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(source, target[:, :-1])
            loss = criterion(
                logits.reshape(-1, VOCAB),
                target[:, 1:].reshape(-1),
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for source, target in val_loader:
            source = source.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            logits = model(source, target[:, :-1])
            val_loss += criterion(
                logits.reshape(-1, VOCAB),
                target[:, 1:].reshape(-1),
            ).item()

    train_loss /= max(1, len(train_loader))
    val_loss /= max(1, len(val_loader))
    print(f'epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f}')

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'model': model.state_dict(),
            'best_val': best_val,
            'config': {
                'vocab': VOCAB,
                'max_len': MAX_LEN,
                'd_model': D_MODEL,
                'nhead': NHEAD,
                'enc_layers': ENC_LAYERS,
                'dec_layers': DEC_LAYERS,
                'ff': FF,
            },
        }, best_path)
        print('saved best:', best_path)


## 8. 推論

コピー対象の反復まで罰してしまうので、旧版の repetition penalty は外します。


In [ ]:
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model'])
model.eval()

@torch.no_grad()
def infer(text, max_new_tokens=None):
    source_ids = encode_full('<NIDA_FICTION> ' + text)
    if len(source_ids) > MAX_LEN:
        raise ValueError(f'input is too long: {len(source_ids)} tokens > {MAX_LEN}')

    source = torch.tensor([source_ids], device=device)
    output = [BOS]
    limit = MAX_LEN - 1 if max_new_tokens is None else min(max_new_tokens, MAX_LEN - 1)

    for _ in range(limit):
        logits = model(source, torch.tensor([output], device=device))[0, -1]
        next_id = int(torch.argmax(logits))

        if next_id == EOS:
            break

        output.append(next_id)

    return sp.decode(output[1:])

tests = [
    '今日は学校です。',
    '明日は雨が降るかもしれません。',
    '最近少し暖かくなってきました。',
    '製品KuchoLM-X7-2026は正常に動作しています。',
    '髙﨑𠮷野家ABC-123を確認しました。',
]
for sample in tests:
    print(sample, '->', infer(sample))


## 9. コピー性能の評価

通常の validation に加えて、未知・低頻度になりやすい `[REF:...]` が **1文字も変わらず残った割合** を測ります。


In [ ]:
def similarity(expected, actual):
    return difflib.SequenceMatcher(None, expected, actual).ratio()

def evaluate(limit=200):
    normal = val_rows[:limit]
    exact = 0
    similarity_total = 0.0

    for tagged_source, expected in normal:
        source = tagged_source.removeprefix('<NIDA_FICTION> ')
        actual = infer(source)
        exact += actual == expected
        similarity_total += similarity(expected, actual)

    marker_ok = 0
    marker_total = min(limit, len(copy_eval_rows))
    for tagged_source, expected, ref in copy_eval_rows[:marker_total]:
        source = tagged_source.removeprefix('<NIDA_FICTION> ')
        actual = infer(source)
        marker_ok += ref in actual

    print(f'exact match: {exact / max(1, len(normal)):.1%}')
    print(f'char similarity: {similarity_total / max(1, len(normal)):.3f}')
    print(f'REF copy accuracy: {marker_ok / max(1, marker_total):.1%}')

evaluate()
print('model:', best_path)
print('tokenizer:', WORK_DIR / 'kucholm_spm.model')
